
# HazardX
 Random Forest and Logistic Regression Model


In [1]:

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split

from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score
)

from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression

from imblearn.over_sampling import SMOTE



## Load Dataset


In [5]:

df = pd.read_csv("../data/IHMStefanini_industrial_safety_and_health_database.csv")

df.head()


,Date,Countries,Local,Industry Sector,Accident Level,Potential Accident Level,Gender,Employee or Third Party,Critical Risk
0,2016-01-01 00:00:00,Country_01,Local_01,Mining,I,IV,Male,Third Party,Pressed
1,2016-01-02 00:00:00,Country_02,Local_02,Mining,I,IV,Male,Employee,Pressurized Systems
2,2016-01-06 00:00:00,Country_01,Local_03,Mining,I,III,Male,Third Party (Remote),Manual Tools
3,2016-01-08 00:00:00,Country_01,Local_04,Mining,I,I,Male,Third Party,Others
4,2016-01-10 00:00:00,Country_01,Local_04,Mining,IV,IV,Male,Third Party,Others


In [6]:

print("Dataset Shape:", df.shape)
print(df.columns)


Dataset Shape: (439, 9)
Index(['Date', 'Countries', 'Local', 'Industry Sector', 'Accident Level',
       'Potential Accident Level', 'Gender', 'Employee or Third Party',
       'Critical Risk'],
      dtype='object')



## Target Engineering

The original dataset contains five accident severity levels:

- I
- II
- III
- IV
- V

Due to heavy class imbalance, we convert this into binary classification:

- 0 = Non-Severe
- 1 = Severe


In [7]:

df["target"] = df["Accident Level"].apply(
    lambda x: 1 if x in ["IV", "V"] else 0
)

print(df["target"].value_counts())


target
0    399
1     40
Name: count, dtype: int64



## Feature Selection


In [38]:

features = [
    "Countries",
    "Local",
    "Industry Sector",
    "Employee or Third Party",
    "Critical Risk"
]

X = df[features].copy()
y = df["target"]



## Encode Categorical Features


In [39]:

encoders = {}

for col in X.columns:
    le = LabelEncoder()
    X[col] = le.fit_transform(X[col].astype(str))
    encoders[col] = le



## Train Test Split


In [40]:

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)



## Handle Class Imbalance Using SMOTE


In [41]:

smote = SMOTE(random_state=42)

X_train_resampled, y_train_resampled = smote.fit_resample(
    X_train,
    y_train
)

print("Before SMOTE:")
print(y_train.value_counts())

print("\nAfter SMOTE:")
print(y_train_resampled.value_counts())


Before SMOTE:
target
0    319
1     32
Name: count, dtype: int64

After SMOTE:
target
0    319
1    319
Name: count, dtype: int64



# Random Forest Model


In [42]:

rf_model = RandomForestClassifier(
    n_estimators=300,
    max_depth=10,
    random_state=42
)

rf_model.fit(X_train_resampled, y_train_resampled)


,n_estimators,300
,criterion,'gini'
,max_depth,10
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,'sqrt'
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [43]:

rf_predictions = rf_model.predict(X_test)

rf_accuracy = accuracy_score(y_test, rf_predictions)
rf_f1 = f1_score(y_test, rf_predictions, average="macro")

print("Random Forest Accuracy:", rf_accuracy)
print("Random Forest Macro F1:", rf_f1)


Random Forest Accuracy: 0.7954545454545454
Random Forest Macro F1: 0.5938461538461539


In [44]:

print(classification_report(y_test, rf_predictions))


              precision    recall  f1-score   support

           0       0.94      0.82      0.88        80
           1       0.22      0.50      0.31         8

    accuracy                           0.80        88
   macro avg       0.58      0.66      0.59        88
weighted avg       0.88      0.80      0.83        88



In [45]:

print(confusion_matrix(y_test, rf_predictions))


[[66 14]
 [ 4  4]]



# Logistic Regression Model


In [46]:

lr_model = LogisticRegression(
    max_iter=5000,
    random_state=42
)

lr_model.fit(X_train_resampled, y_train_resampled)


,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0
,fit_intercept,True
,intercept_scaling,1
,class_weight,None
,random_state,42
,solver,'lbfgs'
,max_iter,5000
,multi_class,'deprecated'


In [47]:

lr_predictions = lr_model.predict(X_test)

lr_accuracy = accuracy_score(y_test, lr_predictions)
lr_f1 = f1_score(y_test, lr_predictions, average="macro")

print("Logistic Regression Accuracy:", lr_accuracy)
print("Logistic Regression Macro F1:", lr_f1)


Logistic Regression Accuracy: 0.6704545454545454
Logistic Regression Macro F1: 0.432258064516129


In [48]:

print(classification_report(y_test, lr_predictions))


              precision    recall  f1-score   support

           0       0.89      0.72      0.80        80
           1       0.04      0.12      0.06         8

    accuracy                           0.67        88
   macro avg       0.47      0.42      0.43        88
weighted avg       0.82      0.67      0.73        88



In [49]:

print(confusion_matrix(y_test, lr_predictions))


[[58 22]
 [ 7  1]]



## Save Model and Encoder


In [ ]:

import pickle
pickle.dump(rf_model, open("random_forest_model.pkl", "wb"))
pickle.dump(encoders, open("encoders.pkl", "wb"))
print("Model saved successfully")



## notes for this version

binary classification is more realistic for our dataset since it has <500 rows; 
additional features improve signal quality; 
macro F1 gives a more honest evaluation metric; 
smote is used to improve minority class learning; 